In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install faiss-cpu transformers==4.44.2 keybert gliner sentence-transformers

In [3]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


DATA_DIR = "/content/drive/MyDrive/research_paper_search/data"

CLEANED_CSV_PATH = os.path.join(DATA_DIR, "cleaned_arxiv_papers.csv")
EMBEDDINGS_PATH  = os.path.join(DATA_DIR, "arxiv_embeddings.npy")
FAISS_INDEX_PATH = os.path.join(DATA_DIR, "paper_faiss.index")

print("Data folder:", DATA_DIR)


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Data folder: /content/drive/MyDrive/research_paper_search/data


In [4]:
print("Loading dataset...")
df = pd.read_csv(CLEANED_CSV_PATH)

print("Loading embeddings...")
embeddings = np.load(EMBEDDINGS_PATH)

print("df shape      :", df.shape)
print("embeddings shape:", embeddings.shape)

# Sanity check — dono ki row count same honi chahiye
assert len(df) == len(embeddings), "❌ Mismatch! df rows aur embeddings count alag hain."
print("✅ Sanity check passed!")


Loading dataset...
Loading embeddings...
df shape      : (50000, 3)
embeddings shape: (50000, 384)
✅ Sanity check passed!


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("✅ Sentence Transformer loaded.")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Sentence Transformer loaded.


In [6]:
import faiss

# ✅ Explicit copy — original embeddings array in-place modify na ho
faiss_embeddings = embeddings.copy().astype("float32")
faiss.normalize_L2(faiss_embeddings)

EMBEDDING_DIM = faiss_embeddings.shape[1]

if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS index...")
    index = faiss.read_index(FAISS_INDEX_PATH)
else:
    print("Creating new FAISS index...")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(faiss_embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print("FAISS index saved to:", FAISS_INDEX_PATH)

print("Total vectors in index:", index.ntotal)


Loading existing FAISS index...
Total vectors in index: 50000


In [7]:
def search_paper(query, k=5):
    """Query ko encode karke FAISS index me top-k similar papers dhoondta hai."""
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    return D, I

# Quick test
D, I = search_paper("deep learning for medical image analysis")
print("Top 5 results:")
for score, idx in zip(D[0], I[0]):
    print(f"  Score: {score:.4f}  |  {df.iloc[idx]['title']}")


Top 5 results:
  Score: 0.7254  |  An overview of deep learning in medical imaging focusing on MRI
  Score: 0.7171  |  Medical Imaging with Deep Learning: MIDL 2019 -- Extended Abstract Track
  Score: 0.7123  |  Deep learning in radiology: an overview of the concepts and a survey of
  the state of the art
  Score: 0.6807  |  A Perspective on Deep Imaging
  Score: 0.6799  |  Deep Learning in Cardiology


In [8]:
from gliner import GLiNER

ner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

TECH_LABELS = ["programming language", "framework", "library", "database", "tool"]
print("✅ GLiNER loaded. Labels:", TECH_LABELS)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✅ GLiNER loaded. Labels: ['programming language', 'framework', 'library', 'database', 'tool']


In [9]:
def extract_tech_entities(text, threshold=0.5):
    """
    GLiNER se tech entities nikalta hai.
    threshold (0-1): confidence cutoff
      - Ghata do (0.3) → zyada entities pakde
      - Badha do (0.7) → sirf high-confidence entities
    """
    if not isinstance(text, str) or not text.strip():
        return {}

    entities = ner_model.predict_entities(text, TECH_LABELS, threshold=threshold)

    found = {}
    for ent in entities:
        found.setdefault(ent["label"], set()).add(ent["text"])

    return {label: sorted(vals) for label, vals in found.items()}

# Quick test — "Bun" aur "DuckDB" kisi manual dictionary me nahi hain
sample_text = ("This paper uses Python and FastAPI for deployment, PyTorch for training. "
               "We also tested Bun as a runtime and DuckDB for analytical queries.")
print(extract_tech_entities(sample_text))


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'programming language': ['Python'], 'library': ['Bun', 'FastAPI'], 'database': ['DuckDB']}


In [10]:
from transformers import pipeline

summarizer_device = 0 if torch.cuda.is_available() else -1  # GPU auto-detect

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=summarizer_device
)
print("✅ Summarizer loaded.")


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Summarizer loaded.


In [11]:
def summarize_text(text, max_length=120, min_length=40):
    """Abstract ko chhoti readable summary me convert karta hai."""
    if not isinstance(text, str) or len(text.split()) < 20:
        return text  # bahut chota text — summarize karne ki zaroorat nahi
    summary = summarizer(text, max_length=max_length, min_length=min_length, truncation=True)
    return summary[0]["summary_text"]


In [12]:
from keybert import KeyBERT

kw_model = KeyBERT(model=model)  # same sentence-transformer reuse
print("✅ KeyBERT loaded.")


✅ KeyBERT loaded.


In [13]:
def extract_keywords(text, top_n=8):
    """Paper ke important keyphrases nikalta hai."""
    return kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=top_n
    )

# Quick test
print(extract_keywords(df.iloc[0]["abstract"]))


[('information theoretic characterization', 0.5784), ('characterization achievable predictor', 0.5384), ('function information theoretic', 0.5281), ('information theoretic', 0.4972), ('distribution allowable predictors', 0.4931), ('loss function information', 0.472), ('admissible predictors', 0.4638), ('conditions admissible predictors', 0.4538)]


In [14]:
def search_engine(query, k=5):
    """
    Full pipeline — ek call me charo outputs:
      1. Semantic Search  (FAISS)
      2. Summarization    (DistilBART)
      3. Keyword Extract  (KeyBERT)
      4. NER              (GLiNER)
    """
    D, I = search_paper(query, k)

    results = []
    for score, idx in zip(D[0], I[0]):
        title    = df.iloc[idx]["title"]
        abstract = df.iloc[idx]["abstract"]

        summary  = summarize_text(abstract)
        keywords = extract_keywords(abstract, top_n=8)
        entities = extract_tech_entities(abstract)

        results.append({
            "score"       : float(score),
            "title"       : title,
            "summary"     : summary,
            "keywords"    : [kw for kw, _ in keywords],
            "tech_entities": entities,
        })

    return results


def print_results(results):
    for i, r in enumerate(results, 1):
        print("=" * 80)
        print(f"Result #{i}")
        print(f"Similarity Score : {r['score']:.4f}")
        print(f"Title            : {r['title']}")
        print(f"Summary          : {r['summary']}")
        print(f"Keywords         : {', '.join(r['keywords'])}")
        if r["tech_entities"]:
            print("Tech Entities:")
            for label, vals in r["tech_entities"].items():
                print(f"   [{label}] {', '.join(vals)}")
        else:
            print("Tech Entities    : None detected")
        print()


In [16]:
results = search_engine("natural language processing for text classification", k=5)
print_results(results)


Your max_length is set to 120, but your input_length is only 99. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=49)


Result #1
Similarity Score : 0.6618
Title            : Text Classification Algorithms: A Survey
Summary          :  Machine learning approaches have achieved surpassing results in natural language processing . The success of these learning algorithms relies on their ability to understand complex models and non-linear relationships within data . Finding suitable structures, architectures, and techniques for text classification is a challenge for researchers .
Keywords         : text classification algorithms, text classification, techniques text classification, text feature extractions, classify texts, accurately classify texts, classify texts applications, overview text classification
Tech Entities    : None detected

Result #2
Similarity Score : 0.6349
Title            : Semi-supervised Classification for Natural Language Processing
Summary          :  Semi-supervised classification exploits only labeled data that are expensive, often difficult to get, inadequate in quantity, and requ